In [309]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import statsmodels.api as sm


In [310]:
start_time = '2025-07-13'
end_time = '2026-07-13'
nifty_50_idd = yf.download('^NSEI',start_time, end_time, interval='1h')
nifty_50_idd.head()

/tmp/ipykernel_792/281414008.py:3: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI
Datetime,,,,,
2025-07-14 03:45:00+00:00,25119.800781,25149.050781,25042.099609,25149.050781,0
2025-07-14 04:45:00+00:00,25063.949219,25127.050781,25061.849609,25120.750000,0
2025-07-14 05:45:00+00:00,25045.400391,25066.400391,25019.800781,25063.750000,0
2025-07-14 06:45:00+00:00,25026.349609,25060.949219,25002.750000,25046.449219,0
2025-07-14 07:45:00+00:00,25046.650391,25058.550781,25012.900391,25027.250000,0


In [311]:
# Create a copy and flatten the MultiIndex columns from yfinance
clean_df = nifty_50_idd.copy()
clean_df.columns = clean_df.columns.get_level_values(0)

# Reset index to turn 'Datetime' into a column
clean_df = clean_df.reset_index()

# Create separate Date and Time columns
clean_df['Date'] = clean_df['Datetime'].dt.date
clean_df['Time'] = clean_df['Datetime'].dt.time
final_columns = ['Date', 'Time', 'Open', 'High', 'Low', 'Close', 'Volume']
final_df = clean_df[final_columns]

display(final_df.head())

Price,Date,Time,Open,High,Low,Close,Volume
0,2025-07-14,03:45:00,25149.050781,25149.050781,25042.099609,25119.800781,0
1,2025-07-14,04:45:00,25120.750000,25127.050781,25061.849609,25063.949219,0
2,2025-07-14,05:45:00,25063.750000,25066.400391,25019.800781,25045.400391,0
3,2025-07-14,06:45:00,25046.449219,25060.949219,25002.750000,25026.349609,0
4,2025-07-14,07:45:00,25027.250000,25058.550781,25012.900391,25046.650391,0


In [312]:
# Combine 'Date' and 'Time' into a single datetime column for plotting
final_df['DateTime'] = pd.to_datetime(final_df['Date'].astype(str) + ' ' + final_df['Time'].astype(str))

# Create an interactive line plot of 'Close' price over time
fig = px.line(final_df, x='DateTime', y='Close', title='Nifty 50 Close Price Over Time')
fig.update_xaxes(rangeslider_visible=True)
fig.show()

In [313]:
# Feature Engineering: Calculate ATR (Average True Range)
high_low = final_df['High'] - final_df['Low']
high_close = np.abs(final_df['High'] - final_df['Close'].shift())
low_close = np.abs(final_df['Low'] - final_df['Close'].shift())

true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
WINDOW_PERIOD = 14
alpha = 1/WINDOW_PERIOD
final_df['ATR'] = true_range.ewm(alpha).mean()

# Calculate log returns
final_df['Log_Returns'] = np.log(final_df['Close'] / final_df['Close'].shift(1))

# Drop NaNs created by rolling ATR and shifts
final_df.dropna(inplace=True)

# Use ATR (normalized by price) and Log Returns as features
final_df['ATR_Norm'] = final_df['ATR'] / final_df['Close']

final_df.head()

Price,Date,Time,Open,High,Low,Close,Volume,DateTime,ATR,Log_Returns,ATR_Norm
1,2025-07-14,04:45:00,25120.750000,25127.050781,25061.849609,25063.949219,0,2025-07-14 04:45:00,67.810547,-0.002226,0.002706
2,2025-07-14,05:45:00,25063.750000,25066.400391,25019.800781,25045.400391,0,2025-07-14 05:45:00,48.007804,-0.000740,0.001917
3,2025-07-14,06:45:00,25046.449219,25060.949219,25002.750000,25026.349609,0,2025-07-14 06:45:00,57.519979,-0.000761,0.002298
4,2025-07-14,07:45:00,25027.250000,25058.550781,25012.900391,25046.650391,0,2025-07-14 07:45:00,46.441682,0.000811,0.001854
5,2025-07-14,08:45:00,25046.849609,25108.500000,25037.199219,25079.849609,0,2025-07-14 08:45:00,69.643510,0.001325,0.002777


## Implementation of Mean reverting strategy, entry only on those trades where mean reversion exists, and exit from the trades when it ceases to exists

In [314]:
def calculate_dynamic_smoothing_factor(df, atr_period=30):

    # Rolling normalization removes lookahead bias
    window = max(200, atr_period)

    atr_min = (
        df["ATR_Norm"]
        .rolling(window=window, min_periods=30)
        .min()
    )

    atr_max = (
        df["ATR_Norm"]
        .rolling(window=window, min_periods=30)
        .max()
    )

    alpha = (
        (df["ATR_Norm"] - atr_min)
        /
        (atr_max - atr_min)
    )

    alpha = (
        alpha
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.50)
        .clip(0.05, 0.95)
    )

    return alpha

In [315]:
def calculate_adaptive_ema(data, dynamic_smoothing_factor):
    """
    Calculates an Adaptive Exponential Moving Average (AEMA).

    Args:
        data (pd.Series): The time series data to smooth (e.g., 'Close' prices).
        dynamic_smoothing_factors (pd.Series): A series of smoothing factors (alpha).

    Returns:
        pd.Series: The calculated Adaptive EMA.
    """
    aema = pd.Series(index=data.index, dtype=float)

    aema.iloc[0] = data.iloc[0] # Initialize the first AEMA value

    for i in range(1, len(data)):
        alpha = dynamic_smoothing_factor.iloc[i]
        aema.iloc[i] = (alpha * data.iloc[i]) + ((1 - alpha) * aema.iloc[i-1])

    return aema



In [316]:
# %%
# Calculate dynamic smoothing factor
final_df['dynamic_smoothing_factor'] = calculate_dynamic_smoothing_factor(final_df)

# Calculate the Adaptive EMA
final_df['AEMA'] = calculate_adaptive_ema(final_df['Close'], final_df['dynamic_smoothing_factor'])

# Calculate AEMA Slope (rate of change)
final_df['AEMA_Slope'] = final_df['AEMA'].diff()

# Display the DataFrame with the new AEMA column
print(final_df[['Date', 'Close', 'dynamic_smoothing_factor', 'AEMA']].head())

Price        Date         Close  dynamic_smoothing_factor          AEMA
1      2025-07-14  25063.949219                       0.5  25063.949219
2      2025-07-14  25045.400391                       0.5  25054.674805
3      2025-07-14  25026.349609                       0.5  25040.512207
4      2025-07-14  25046.650391                       0.5  25043.581299
5      2025-07-14  25079.849609                       0.5  25061.715454


In [317]:
# %%
final_df['rolling_z_score'] = (final_df['Close'] - final_df['AEMA']) / final_df['AEMA'].std()

# Prepare data for OU model: Delta Z vs Lagged Z
z_lag = final_df['rolling_z_score'].shift(1)
z_diff = final_df['rolling_z_score'].diff()

# Drop NaNs
ou_data = pd.DataFrame({'z_lag': z_lag, 'z_diff': z_diff}).dropna()

# Run OLS Regression: dz = lambda * z_t-1 + intercept
# We use the slope (lambda) to find the speed of mean reversion
model_ou = sm.OLS(ou_data['z_diff'], ou_data['z_lag'])
results_ou = model_ou.fit()
lambda_val = results_ou.params.iloc[0]

# Calculate Half-Life (in hours)
# Note: lambda must be negative for a mean-reverting process
if lambda_val < 0:
    half_life = -np.log(2) / lambda_val
    print(f"OU Lambda (Speed of Reversion): {lambda_val:.4f}")
    print(f"Estimated Half-Life: {half_life:.2f} hours")
else:
    print(f"Warning: Lambda is positive ({lambda_val:.4f}), indicating a trending (non-reverting) series in this window.")

# Store in final_df for potential rolling analysis
final_df['ou_lambda'] = lambda_val

OU Lambda (Speed of Reversion): -0.1370
Estimated Half-Life: 5.06 hours


In [318]:
# %%
# Use a rolling window related to the calculated half-life (~4 hours)
HL_WINDOW = int(round(half_life))

# Recalculate rolling Z-score with optimized window
alpha = 2/(HL_WINDOW + 1)
rolling_std = (final_df['Close'] - final_df['AEMA']).ewm(alpha).std()
final_df['rolling_z_score'] = (final_df['Close'] - final_df['AEMA']) / rolling_std

# ==========================================================
# MEAN REVERSION WITH ATR STOP
# ==========================================================

ENTRY_Z = 1.0
EXIT_Z = 0.2
ATR_MULT = 2.0
MAX_HOLD = 8       # hours

position = 0
entry_price = 0
bars_in_trade = 0

positions = []

for i in range(len(final_df)):

    price = final_df["Close"].iloc[i]
    z = final_df["rolling_z_score"].iloc[i]
    atr = final_df["ATR"].iloc[i]

    # -----------------------------------------
    # NO POSITION
    # -----------------------------------------

    if position == 0:

        if z > ENTRY_Z:

            position = 1
            entry_price = price
            bars_in_trade = 0

        elif z < -ENTRY_Z:

            position = -1
            entry_price = price
            bars_in_trade = 0

    # -----------------------------------------
    # LONG POSITION
    # -----------------------------------------

    elif position == 1:

        bars_in_trade += 1

        stop_price = entry_price - ATR_MULT * atr

        if (
            price <= stop_price
            or z < EXIT_Z
            or bars_in_trade >= MAX_HOLD
        ):
            position = 0

    # -----------------------------------------
    # SHORT POSITION
    # -----------------------------------------

    elif position == -1:

        bars_in_trade += 1

        stop_price = entry_price + ATR_MULT * atr

        if (
            price >= stop_price
            or z > -EXIT_Z
            or bars_in_trade >= MAX_HOLD
        ):
            position = 0

    positions.append(position)

final_df["mean_reverting_position"] = (
    pd.Series(positions, index=final_df.index)
    .shift(1)
    .fillna(0)
)

print(final_df[['Date', 'Close', 'AEMA', 'rolling_z_score', 'mean_reverting_position']].tail())

Price        Date         Close          AEMA  rolling_z_score  \
1685   2026-07-10  24164.750000  24135.358415         1.345630   
1686   2026-07-10  24185.800781  24138.690109         3.383422   
1687   2026-07-10  24186.449219  24142.640404         6.291908   
1688   2026-07-10  24208.650391  24147.706508         4.990097   
1689   2026-07-10  24211.650391  24150.903702         9.047351   

Price  mean_reverting_position  
1685                       1.0  
1686                       1.0  
1687                       1.0  
1688                       1.0  
1689                       1.0  


In [319]:
# Recalculate strategy returns based on updated positions
final_df['hourly_return'] = final_df['Close'].pct_change()
target_vol = final_df["ATR"].rolling(100).median()

vol_scalar = (
    target_vol /
    final_df["ATR"]
).clip(0.5, 2.0)

final_df["position_scaled"] = (
    final_df["mean_reverting_position"]
    * vol_scalar.shift(1)
)

target_vol = final_df["ATR"].rolling(100).median()

vol_scalar = (
    target_vol /
    final_df["ATR"]
).clip(0.5,2.0)

final_df["position_scaled"] = (
    final_df["mean_reverting_position"]
    * vol_scalar.shift(1)
)

final_df["mean_reverting_return"] = (
    final_df["hourly_return"]
    * final_df["position_scaled"]
)
final_df['mean_reverting_cumulative_return'] = (1 + final_df['mean_reverting_return']).cumprod() - 1

# Plot the cumulative returns
fig = go.Figure()
fig.add_trace(go.Scatter(x=final_df['Date'], y=final_df['mean_reverting_cumulative_return'], mode='lines', name='Mean Reverting Cumulative Return', line=dict(color='green')))
fig.update_layout(title='Mean Reverting Cumulative Returns (Gross)', xaxis_title='Date', yaxis_title='Cumulative Return')
fig.show()

In [320]:
from numpy._core.multiarray import min_scalar_type
# Assuming 6.5 trading hours per day and 252 trading days per year for annualization
TRADING_HOURS_PER_YEAR = 6.5 * 252

# Calculate Annualized Return
# Use the product of (1 + daily_return) for compounding, then annualize based on the number of data points
# A more robust way to annualize is to get the mean hourly return and raise it to the power of trading hours per year
mr_annualized_return = (1 + final_df['mean_reverting_return']).mean()**TRADING_HOURS_PER_YEAR - 1

# Calculate Max Drawdown
cumulative_returns = (1 + final_df['mean_reverting_return']).cumprod()
peak = cumulative_returns.expanding(min_periods=1).max()
drawdown = (cumulative_returns / peak) - 1
max_drawdown = drawdown.min()

# Calculate Annualized Standard Deviation of Returns
mr_annualized_std_dev = final_df['mean_reverting_return'].std() * np.sqrt(TRADING_HOURS_PER_YEAR)

# Calculate Sharpe Ratio (assuming risk-free rate = 0 for simplicity)
mr_sharpe_ratio = mr_annualized_return / mr_annualized_std_dev

# Calculate Downside Deviation for Sortino Ratio
downside_returns = final_df[final_df['mean_reverting_return'] < 0]['mean_reverting_return']
annualized_downside_std_dev = downside_returns.std() * np.sqrt(TRADING_HOURS_PER_YEAR)

# Calculate Sortino Ratio (assuming risk-free rate = 0)
sortino_ratio = mr_annualized_return / annualized_downside_std_dev if annualized_downside_std_dev != 0 else np.nan

# Calculate Calmar Ratio
calmar_ratio = mr_annualized_return / abs(max_drawdown) if max_drawdown != 0 else np.nan

final_df['mean_reverting_trade_id'] = (final_df['mean_reverting_position'] != final_df['mean_reverting_position'].shift()).cumsum()

# Filter only non-zero position periods to analyze actual trades
trades = final_df[final_df['mean_reverting_position'] != 0].groupby('mean_reverting_trade_id').agg({
    'mean_reverting_return': lambda x: (1 + x).prod() - 1,
    'Date': 'first'
})

num_trades = len(trades)
pos_trades = trades[trades['mean_reverting_return'] > 0]
neg_trades = trades[trades['mean_reverting_return'] <= 0]

num_pos_trades = len(pos_trades)
num_neg_trades = len(neg_trades)
hit_ratio = num_pos_trades / num_trades if num_trades > 0 else 0

# Information Ratio Calculation
benchmark_return = final_df['hourly_return']
active_return = final_df['mean_reverting_return'] - benchmark_return
tracking_error = active_return.std() * np.sqrt(TRADING_HOURS_PER_YEAR)
annualized_active_return = active_return.mean() * TRADING_HOURS_PER_YEAR
information_ratio = annualized_active_return / tracking_error if tracking_error != 0 else np.nan

# Display Results
print(f"--- Trade Statistics for Mean reverting ---")
print(f"Total Number of Trades: {num_trades}")
print(f"Number of Positive Trades: {num_pos_trades}")
print(f"Number of Negative Trades: {num_neg_trades}")
print(f"Hit Ratio: {hit_ratio:.2%}")
print(f"Information Ratio: {information_ratio:.2f}")

print(f"Annualized Return: {mr_annualized_return:.2%}")
print(f"Max Drawdown: {max_drawdown:.2%}")
print(f"Sharpe Ratio: {mr_sharpe_ratio:.2f}")
print(f"Sortino Ratio: {sortino_ratio:.2f}")
print(f"Calmar Ratio: {calmar_ratio:.2f}")

--- Trade Statistics for Mean reverting ---
Total Number of Trades: 267
Number of Positive Trades: 103
Number of Negative Trades: 164
Hit Ratio: 38.58%
Information Ratio: 1.59
Annualized Return: 41.17%
Max Drawdown: -7.69%
Sharpe Ratio: 2.25
Sortino Ratio: 2.65
Calmar Ratio: 5.36


## Intraday directional strategy

In [321]:
# %%
# Generate trading signals
final_df['signal'] = 0

# Buy only if Close > AEMA which represents bullish momentum signal
final_df.loc[(final_df['Close'] > final_df['AEMA']), 'signal'] = 1

# Sell only if Close < AEMA which represents bearish momentum signal
final_df.loc[(final_df['Close'] < final_df['AEMA']), 'signal'] = -1

# Calculate positions: 1 for long, -1 for short, 0 for cash, the shift(1) ensures we trade on the next hour's open
final_df['position'] = final_df['signal'].shift(1).fillna(0)

print(final_df[['Date', 'Time', 'Close', 'AEMA', 'signal', 'position']].head())

Price        Date      Time         Close          AEMA  signal  position
1      2025-07-14  04:45:00  25063.949219  25063.949219       0       0.0
2      2025-07-14  05:45:00  25045.400391  25054.674805      -1       0.0
3      2025-07-14  06:45:00  25026.349609  25040.512207      -1      -1.0
4      2025-07-14  07:45:00  25046.650391  25043.581299       1      -1.0
5      2025-07-14  08:45:00  25079.849609  25061.715454       1       1.0


In [322]:
# Recalculate strategy returns based on updated positions
final_df['hourly_return'] = final_df['Close'].pct_change()
final_df['tr_strategy_return'] = final_df['hourly_return'] * final_df['position']
final_df['cumulative_strategy_return'] = (1 + final_df['tr_strategy_return']).cumprod() - 1

# Plot the cumulative returns
fig = go.Figure()
fig.add_trace(go.Scatter(x=final_df['Date'], y=final_df['cumulative_strategy_return'], mode='lines', name='Strategy Cumulative Return', line=dict(color='green')))
fig.update_layout(title='Trend Strategy Cumulative Returns (Gross)', xaxis_title='Date', yaxis_title='Cumulative Return')
fig.show()

In [323]:
# Assuming 6.5 trading hours per day and 252 trading days per year for annualization
TRADING_HOURS_PER_YEAR = 6.5 * 252

# Calculate Annualized Return
# Use the product of (1 + daily_return) for compounding, then annualize based on the number of data points
# A more robust way to annualize is to get the mean hourly return and raise it to the power of trading hours per year
tr_annualized_return = (1 + final_df['tr_strategy_return']).mean()**TRADING_HOURS_PER_YEAR - 1

# Calculate Max Drawdown
cumulative_returns = (1 + final_df['tr_strategy_return']).cumprod()
peak = cumulative_returns.expanding(min_periods=1).max()
drawdown = (cumulative_returns / peak) - 1
max_drawdown = drawdown.min()

# Calculate Annualized Standard Deviation of Returns
tr_annualized_std_dev = final_df['tr_strategy_return'].std() * np.sqrt(TRADING_HOURS_PER_YEAR)

# Calculate Sharpe Ratio (assuming risk-free rate = 0 for simplicity)
tr_sharpe_ratio = tr_annualized_return / tr_annualized_std_dev

# Calculate Downside Deviation for Sortino Ratio
downside_returns = final_df[final_df['tr_strategy_return'] < 0]['tr_strategy_return']
annualized_downside_std_dev = downside_returns.std() * np.sqrt(TRADING_HOURS_PER_YEAR)

# Calculate Sortino Ratio (assuming risk-free rate = 0)
tr_sortino_ratio = tr_annualized_return / annualized_downside_std_dev if annualized_downside_std_dev != 0 else np.nan

# Calculate Calmar Ratio
tr_calmar_ratio = tr_annualized_return / abs(max_drawdown) if max_drawdown != 0 else np.nan

final_df['trade_id'] = (final_df['position'] != final_df['position'].shift()).cumsum()

# Filter only non-zero position periods to analyze actual trades
trades = final_df[final_df['position'] != 0].groupby('trade_id').agg({
    'tr_strategy_return': lambda x: (1 + x).prod() - 1,
    'Date': 'first'
})

num_trades = len(trades)
pos_trades = trades[trades['tr_strategy_return'] > 0]
neg_trades = trades[trades['tr_strategy_return'] <= 0]

num_pos_trades = len(pos_trades)
num_neg_trades = len(neg_trades)
hit_ratio = num_pos_trades / num_trades if num_trades > 0 else 0

# Information Ratio Calculation
benchmark_return = final_df['hourly_return']
active_return = final_df['tr_strategy_return'] - benchmark_return
tracking_error = active_return.std() * np.sqrt(TRADING_HOURS_PER_YEAR)
annualized_active_return = active_return.mean() * TRADING_HOURS_PER_YEAR
information_ratio = annualized_active_return / tracking_error if tracking_error != 0 else np.nan

# Display Results
print(f"--- Trade Statistics for Directional Trend ---")
print(f"Total Number of Trades: {num_trades}")
print(f"Number of Positive Trades: {num_pos_trades}")
print(f"Number of Negative Trades: {num_neg_trades}")
print(f"Hit Ratio: {hit_ratio:.2%}")
print(f"Information Ratio: {information_ratio:.2f}")

print(f"Annualized Return: {tr_annualized_return:.2%}")
print(f"Max Drawdown: {max_drawdown:.2%}")
print(f"Sharpe Ratio: {tr_sharpe_ratio:.2f}")
print(f"Sortino Ratio: {tr_sortino_ratio:.2f}")
print(f"Calmar Ratio: {tr_calmar_ratio:.2f}")

--- Trade Statistics for Directional Trend ---
Total Number of Trades: 292
Number of Positive Trades: 93
Number of Negative Trades: 199
Hit Ratio: 31.85%
Information Ratio: 0.99
Annualized Return: 17.20%
Max Drawdown: -8.10%
Sharpe Ratio: 1.36
Sortino Ratio: 1.84
Calmar Ratio: 2.12


## Intraday semi-directional strategy

In [324]:
final_df["semi_signal"] = 0

signals = []
position = 0

ENTRY_Z = 2.0
EXIT_Z = 0.3

for i in range(len(final_df)):

    z = final_df["rolling_z_score"].iloc[i]
    close = final_df["Close"].iloc[i]
    aema = final_df["AEMA"].iloc[i]


    trend_up = (
        close > aema
        and final_df["AEMA_Slope"].iloc[i] > 0

    )

    trend_down = (
        close < aema
        and final_df["AEMA_Slope"].iloc[i] < 0


    )
    # ---------------- ENTRY ----------------


    trend_strength = abs(close - aema) / final_df["ATR"].iloc[i]

    if position == 0:

        if (
            trend_up
            and z > ENTRY_Z
            and trend_strength > 1.2
        ):
            position = 1

        elif (
            trend_down
            and z < -ENTRY_Z
            and trend_strength > 1.2
        ):
            position = -1

    # ---------------- LONG EXIT ----------------

    elif position == 1:

        if (z < EXIT_Z) or trend_down:
            position = 0

    # ---------------- SHORT EXIT ----------------

    elif position == -1:

        if (z > -EXIT_Z) or trend_up:
            position = 0

    signals.append(position)

final_df["semi_position"] = (
    pd.Series(signals, index=final_df.index)
    .shift(1)
    .fillna(0)
)
final_df["semi_position"] = (
    pd.Series(signals, index=final_df.index)
    .shift(1)
    .fillna(0)
)

print("Semi trades:",
      (final_df["semi_position"].diff().abs() > 0).sum())
final_df['dynamic_smoothing_factor'] = calculate_dynamic_smoothing_factor(final_df)
final_df.head()

Semi trades: 209


Price,Date,Time,Open,High,Low,Close,Volume,DateTime,ATR,Log_Returns,...,mean_reverting_return,mean_reverting_cumulative_return,mean_reverting_trade_id,signal,position,tr_strategy_return,cumulative_strategy_return,trade_id,semi_signal,semi_position
1,2025-07-14,04:45:00,25120.750000,25127.050781,25061.849609,25063.949219,0,2025-07-14 04:45:00,67.810547,-0.002226,...,NaN,NaN,1,0,0.0,NaN,NaN,1,0,0.0
2,2025-07-14,05:45:00,25063.750000,25066.400391,25019.800781,25045.400391,0,2025-07-14 05:45:00,48.007804,-0.000740,...,NaN,NaN,1,-1,0.0,-0.000000,0.000000,1,0,0.0
3,2025-07-14,06:45:00,25046.449219,25060.949219,25002.750000,25026.349609,0,2025-07-14 06:45:00,57.519979,-0.000761,...,NaN,NaN,2,-1,-1.0,0.000761,0.000761,2,0,0.0
4,2025-07-14,07:45:00,25027.250000,25058.550781,25012.900391,25046.650391,0,2025-07-14 07:45:00,46.441682,0.000811,...,NaN,NaN,2,1,-1.0,-0.000811,-0.000051,2,0,0.0
5,2025-07-14,08:45:00,25046.849609,25108.500000,25037.199219,25079.849609,0,2025-07-14 08:45:00,69.643510,0.001325,...,NaN,NaN,3,1,1.0,0.001325,0.001274,3,0,0.0


In [325]:
# Calculate Results
final_df['semi_return'] = final_df['hourly_return'] * final_df['semi_position']
final_df['semi_cumulative_return'] = (1 + final_df['semi_return'].fillna(0)).cumprod() - 1

trades_count = final_df['semi_position'].diff().abs().sum()
print(f"Total trades executed: {trades_count}")
print(final_df[['Date', 'Close', 'AEMA', 'rolling_z_score', 'semi_position', 'semi_cumulative_return']].tail())

Total trades executed: 209.0
Price        Date         Close          AEMA  rolling_z_score  semi_position  \
1685   2026-07-10  24164.750000  24135.358415         1.345630            0.0   
1686   2026-07-10  24185.800781  24138.690109         3.383422            0.0   
1687   2026-07-10  24186.449219  24142.640404         6.291908            0.0   
1688   2026-07-10  24208.650391  24147.706508         4.990097            0.0   
1689   2026-07-10  24211.650391  24150.903702         9.047351            1.0   

Price  semi_cumulative_return  
1685                 0.045518  
1686                 0.045518  
1687                 0.045518  
1688                 0.045518  
1689                 0.045647  


In [326]:
# Calculate Performance Metrics for Semi-Directional
TRADING_HOURS_PER_YEAR = 6.5 * 252

# Ensure we drop NaNs for the calculation
semi_returns_series = final_df['semi_return'].dropna()

ann_ret_semi = (1 + semi_returns_series.mean())**TRADING_HOURS_PER_YEAR - 1
std_semi = semi_returns_series.std() * np.sqrt(TRADING_HOURS_PER_YEAR)
sharpe_semi = ann_ret_semi / std_semi if std_semi != 0 else 0

# Calculate cumulative returns for plotting (starts from 0)
cum_ret_semi = (1 + final_df['semi_return'].fillna(0)).cumprod() - 1

# For Max Drawdown, we need the equity curve (starting from 1)
equity_curve_semi = (1 + final_df['semi_return'].fillna(0)).cumprod()
peak_semi = equity_curve_semi.expanding(min_periods=1).max()
ddraw_semi = (equity_curve_semi / peak_semi) - 1
max_dd_semi = ddraw_semi.min()

# Filter only non-zero position periods to analyze actual trades
semi_trades = final_df[final_df['semi_position'] != 0].groupby(final_df['semi_position'].diff().fillna(0).abs().cumsum()).agg({
    'semi_return': lambda x: (1 + x).prod() - 1,
    'Date': 'first'
})

pos_semi_trades = semi_trades[semi_trades['semi_return'] > 0]
neg_semi_trades = semi_trades[semi_trades['semi_return'] <= 0]

num_pos_semi_trades = len(pos_semi_trades)
num_neg_semi_trades = len(neg_semi_trades)
num_semi_trades = num_pos_semi_trades + num_neg_semi_trades
hit_ratio_semi = num_pos_semi_trades / num_semi_trades if num_semi_trades > 0 else 0

# Display Results
print(f"--- Semi-Directional Strategy Statistics ---")
print(f"Total Trades: {num_semi_trades}")
print(f"Number of Positive Trades: {num_pos_semi_trades}")
print(f"Number of Negative Trades: {num_neg_semi_trades}")
print(f"Hit Ratio: {hit_ratio_semi:.2%}")
print(f"Annualized Return: {ann_ret_semi:.2%}")
print(f"Sharpe Ratio: {sharpe_semi:.2f}")
print(f"Max Drawdown: {max_dd_semi:.2%}")

# Visual comparison
fig = go.Figure()
fig.add_trace(go.Scatter(x=final_df['Date'], y=final_df['mean_reverting_cumulative_return'], name='Pure Mean Reversion'))
fig.add_trace(go.Scatter(x=final_df['Date'], y=final_df['cumulative_strategy_return'], name='Pure Trend Following (AEMA)'))
fig.add_trace(go.Scatter(x=final_df['Date'], y=final_df['semi_cumulative_return'], name='Semi-Directional (Hybrid)', line=dict(width=3, color='black')))

fig.update_layout(title='Nifty 50: Strategy Comparison (Hourly)', xaxis_title='Date', yaxis_title='Cumulative Return', template='plotly_dark')
fig.show()

--- Semi-Directional Strategy Statistics ---
Total Trades: 105
Number of Positive Trades: 36
Number of Negative Trades: 69
Hit Ratio: 34.29%
Annualized Return: 4.93%
Sharpe Ratio: 0.50
Max Drawdown: -8.38%


## Portfolio Construction & Multi-Strategy Analysis
In this section, we combine the three signals into an ensemble portfolio. We apply **Volatility Scaling** to ensure that no single strategy (like the high-volatility Pure Trend) dominates the risk profile, thereby optimizing the Calmar Ratio.

In [327]:
# Aggregate Strategy Returns
# ==========================================================
# ENSEMBLE PORTFOLIO (Risk Parity)
# ==========================================================

portfolio_df = pd.DataFrame(index=final_df.index)

portfolio_df["Trend"] = final_df["tr_strategy_return"].fillna(0)
portfolio_df["MR"] = final_df["mean_reverting_return"].fillna(0)
portfolio_df["Semi"] = final_df["semi_return"].fillna(0)

# -------------------------------------------------------
# Rolling Volatility
# -------------------------------------------------------

VOL_WINDOW = 30

rolling_vol = (
    portfolio_df[
        ["Trend","MR","Semi"]
    ]
    .rolling(VOL_WINDOW)
    .std()
)

rolling_vol = rolling_vol.replace(0,np.nan)

# -------------------------------------------------------
# Inverse Volatility Weights
# -------------------------------------------------------

rolling_sharpe = (
    portfolio_df[
        ["Trend","MR","Semi"]
    ]
    .rolling(60)
    .mean()
    /
    portfolio_df[
        ["Trend","MR","Semi"]
    ]
    .rolling(60)
    .std()
)

score = rolling_sharpe / rolling_vol

score = score.clip(lower=0)

weights = score.div(
    score.sum(axis=1),
    axis=0
)

weights = weights.shift(1).fillna(1/3)

weights = weights.div(
    weights.sum(axis=1),
    axis=0
)

weights = weights.shift(1)

weights = weights.fillna(1/3)

portfolio_df["Trend_w"] = weights["Trend"]
portfolio_df["MR_w"] = weights["MR"]
portfolio_df["Semi_w"] = weights["Semi"]

# -------------------------------------------------------
# Portfolio Return
# -------------------------------------------------------

portfolio_df["Portfolio_Return"] = (

      portfolio_df["Trend"]*portfolio_df["Trend_w"]

    + portfolio_df["MR"]*portfolio_df["MR_w"]

    + portfolio_df["Semi"]*portfolio_df["Semi_w"]

)

portfolio_df["Portfolio_Return"] = (
    portfolio_df["Portfolio_Return"]
    .fillna(0)
)

portfolio_df["Portfolio_Cumulative"] = (
    (1+portfolio_df["Portfolio_Return"]).cumprod()-1
)

def performance(ret):

    ann_return=(1+ret.mean())**TRADING_HOURS_PER_YEAR-1

    ann_std=ret.std()*np.sqrt(TRADING_HOURS_PER_YEAR)

    sharpe=ann_return/ann_std if ann_std!=0 else np.nan

    equity=(1+ret).cumprod()

    peak=equity.cummax()

    dd=equity/peak-1

    maxdd=dd.min()

    downside=ret[ret<0].std()*np.sqrt(TRADING_HOURS_PER_YEAR)

    sortino=ann_return/downside if downside!=0 else np.nan

    calmar=ann_return/abs(maxdd) if maxdd!=0 else np.nan

    return ann_return,sharpe,sortino,maxdd,calmar
# Final Combined Visualization
fig = go.Figure()
fig.add_trace(go.Scatter(x=final_df['Date'], y=final_df['cumulative_strategy_return'], name='Trend (AEMA)'))
fig.add_trace(go.Scatter(x=final_df['Date'], y=final_df['mean_reverting_cumulative_return'], name='Mean Reversion'))
fig.add_trace(go.Scatter(x=final_df['Date'], y=final_df['semi_cumulative_return'], name='Semi-Directional'))
fig.add_trace(go.Scatter(x=final_df['Date'], y=portfolio_df['Portfolio_Cumulative'],
                         name='ENCORE Scaled Portfolio', line=dict(width=4, color='gold')))

fig.update_layout(title='Nifty 50: Individual Strategies vs. Combined Scaled Portfolio',
                  xaxis_title='Date', yaxis_title='Cumulative Return', template='plotly_dark')
fig.show()

### **Strategy Risk Comparison: Rolling Volatility**
This plot displays the 30-period rolling annualized volatility for each strategy. Lower and more stable volatility typically leads to higher risk-adjusted returns (Sharpe and Calmar ratios).

In [328]:
# %%
# Calculate 30-period rolling volatility for each strategy component
VOL_WINDOW = 30

rolling_vol_trend = final_df['tr_strategy_return'].rolling(window=VOL_WINDOW).std() * np.sqrt(TRADING_HOURS_PER_YEAR)
rolling_vol_mr = final_df['mean_reverting_return'].rolling(window=VOL_WINDOW).std() * np.sqrt(TRADING_HOURS_PER_YEAR)
rolling_vol_semi = final_df['semi_return'].rolling(window=VOL_WINDOW).std() * np.sqrt(TRADING_HOURS_PER_YEAR)
rolling_vol_port = portfolio_df['Portfolio_Return'].rolling(window=VOL_WINDOW).std() * np.sqrt(TRADING_HOURS_PER_YEAR)

# Plotting
fig_vol = go.Figure()
fig_vol.add_trace(go.Scatter(x=final_df['Date'], y=rolling_vol_trend, name='Trend Volatility', line=dict(dash='dot')))
fig_vol.add_trace(go.Scatter(x=final_df['Date'], y=rolling_vol_mr, name='Mean Reversion Volatility', line=dict(dash='dot')))
fig_vol.add_trace(go.Scatter(x=final_df['Date'], y=rolling_vol_semi, name='Semi-Directional Volatility', line=dict(width=2)))
fig_vol.add_trace(go.Scatter(x=final_df['Date'], y=rolling_vol_port, name='Ensemble Portfolio Volatility', line=dict(width=4, color='gold')))

fig_vol.update_layout(
    title='Strategy Risk Comparison: Rolling Annualized Volatility (30-Period)',
    xaxis_title='Date',
    yaxis_title='Annualized Volatility',
    template='plotly_dark',
    yaxis=dict(tickformat='.1%')
)
fig_vol.show()

REGIME_THRESHOLD = 1.2

use_trend = trend_strength > REGIME_THRESHOLD

portfolio_df["Regime_Return"] = np.where(
    use_trend,
    final_df["tr_strategy_return"],
    final_df["mean_reverting_return"]
)
regime = performance(portfolio_df["Regime_Return"])
print("Regime Sharpe:", regime[1])
print("Regime Calmar:", regime[4])
portfolio_return = np.where(
    use_trend,
    final_df["tr_strategy_return"],
    final_df["mean_reverting_return"]
)
trend=performance(portfolio_df["Trend"])

mr=performance(portfolio_df["MR"])

semi=performance(portfolio_df["Semi"])

ensemble=performance(portfolio_df["Portfolio_Return"])

summary=pd.DataFrame({

"Strategy":[
"Trend",
"Mean Reversion",
"Semi Directional",
"Ensemble"
],

"Annual Return":[
trend[0],
mr[0],
semi[0],
ensemble[0]
],

"Sharpe":[
trend[1],
mr[1],
semi[1],
ensemble[1]
],

"Sortino":[
trend[2],
mr[2],
semi[2],
ensemble[2]
],

"MaxDD":[
trend[3],
mr[3],
semi[3],
ensemble[3]
],

"Calmar":[
trend[4],
mr[4],
semi[4],
ensemble[4]
]

})

print(summary)
fig=go.Figure()

fig.add_trace(go.Scatter(
    x=final_df["Date"],
    y=final_df["cumulative_strategy_return"],
    name="Trend"
))

fig.add_trace(go.Scatter(
    x=final_df["Date"],
    y=final_df["mean_reverting_cumulative_return"],
    name="MR"
))

fig.add_trace(go.Scatter(
    x=final_df["Date"],
    y=final_df["semi_cumulative_return"],
    name="Semi"
))

fig.add_trace(go.Scatter(
    x=final_df["Date"],
    y=portfolio_df["Portfolio_Cumulative"],
    name="Ensemble",
    line=dict(width=4,color="gold")
))

fig.show()
portfolio_df["Regime_Return"] = np.where(
    use_trend,
    final_df["tr_strategy_return"],
    final_df["mean_reverting_return"]
)
regime = performance(portfolio_df["Regime_Return"])

print(
    "Regime Sharpe:",
    regime[1]
)

Regime Sharpe: 1.3576551340148204
Regime Calmar: 2.123305212755646
           Strategy  Annual Return    Sharpe   Sortino     MaxDD    Calmar
0             Trend       0.171888  1.357188  1.840525 -0.081005  2.121946
1    Mean Reversion       0.383213  2.160581  2.462873 -0.076879  4.984632
2  Semi Directional       0.049230  0.503344  0.454923 -0.083765  0.587719
3          Ensemble       0.233826  1.779962  2.293953 -0.064240  3.639890


Regime Sharpe: 1.3576551340148204
